# 01 — Exploratory Data Analysis

Analyse the merged prompt dataset: class distribution, text lengths, and sample examples per threat class.

**Run after:** `python data/download.py && python data/prepare.py`

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

splits = {
    s: [json.loads(line) for line in (Path("data/processed") / f"{s}.jsonl").read_text().splitlines()]
    for s in ["train", "val", "test"]
}
dfs = {s: pd.DataFrame(rows) for s, rows in splits.items()}
print({s: len(df) for s, df in dfs.items()})

## Confusion Matrix (test set)

In [ ]:
import numpy as np
import seaborn as sns
from firewall.classifier.evaluate import evaluate

label_names = ["benign", "injection", "jailbreak", "exfiltration", "escalation"]
results = evaluate("models/classifier", "data/processed/test.jsonl", label_names)

cm = np.array(results["confusion_matrix"])
# Normalize each row by its sum so colors reflect recall (relative scale)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm_norm,
    annot=cm,
    fmt="d",
    cmap="Blues",
    vmin=0.0,
    vmax=1.0,
    xticklabels=label_names,
    yticklabels=label_names,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"shrink": 0.8},
    ax=ax,
)
ax.set(xlabel="Predicted", ylabel="True", title="Test Set Confusion Matrix")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("reports/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nAccuracy: {results['accuracy']:.4f}")
print(f"F1 macro: {results['f1_macro']:.4f}")
print(f"\n{results['classification_report']}")

## Text length distribution

In [ ]:
df_all = pd.concat(dfs.values(), ignore_index=True)
df_all["n_chars"] = df_all["text"].str.len()
df_all["n_words"] = df_all["text"].str.split().str.len()
display(df_all.groupby("label")[["n_chars", "n_words"]].describe().round(1))

## Sample examples per class

In [ ]:
for label in sorted(dfs["train"]["label"].unique()):
    sample = dfs["train"][dfs["train"]["label"] == label].sample(2, random_state=0)
    print(f"\n=== {label} ===")
    for _, row in sample.iterrows():
        print(f"  {row['text'][:120]!r}")